<a href="https://colab.research.google.com/github/havisha07/DeepDTA-Extended/blob/main/DeepDTA_baselines.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!nvidia-smi

Wed Oct  7 14:29:28 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   56C    P8             14W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install -q tensorflow numpy pandas scipy scikit-learn matplotlib

In [3]:
import os
os.chdir('/content')
!rm -rf DeepDTA
!git clone https://github.com/hkmztrk/DeepDTA.git
!echo "===== data folder ====="
!ls -la /content/DeepDTA/data

Cloning into 'DeepDTA'...
remote: Enumerating objects: 442, done.
remote: Counting objects: 100% (44/44), done.
remote: Compressing objects: 100% (15/15), done.
remote: Total 442 (delta 33), reused 32 (delta 29), pack-reused 398 (from 1)
Receiving objects: 100% (442/442), 10.99 MiB | 12.27 MiB/s, done.
Resolving deltas: 100% (174/174), done.
===== data folder =====
total 20
drwxr-xr-x 4 root root 4096 Oct  7 14:29 .
drwxr-xr-x 7 root root 4096 Oct  7 14:29 ..
drwxr-xr-x 3 root root 4096 Oct  7 14:29 davis
drwxr-xr-x 3 root root 4096 Oct  7 14:29 kiba
-rw-r--r-- 1 root root 3454 Oct  7 14:29 README.md


In [4]:
!echo "===== Davis folder ====="
!ls -la /content/DeepDTA/data/davis
!echo ""
!echo "===== First 3 lines of Y ====="
!head -3 /content/DeepDTA/data/davis/Y
!echo ""
!echo "===== Ligands file (first 300 chars) ====="
!head -c 300 /content/DeepDTA/data/davis/ligands_can.txt
!echo ""
!echo ""
!echo "===== Proteins file (first 300 chars) ====="
!head -c 300 /content/DeepDTA/data/davis/proteins.txt

===== Davis folder =====
total 1764
drwxr-xr-x 3 root root   4096 Oct  7 14:29 .
drwxr-xr-x 4 root root   4096 Oct  7 14:29 ..
-rw-r--r-- 1 root root  26892 Oct  7 14:29 drug-drug_similarities_2D.txt
-rw-r--r-- 1 root root 164306 Oct  7 14:29 drug-target_interaction_affinities_Kd__Davis_et_al.2011v1.txt
drwxr-xr-x 2 root root   4096 Oct  7 14:29 folds
-rw-r--r-- 1 root root   5306 Oct  7 14:29 ligands_can.txt
-rw-r--r-- 1 root root   5452 Oct  7 14:29 ligands_iso.txt
-rw-r--r-- 1 root root 355211 Oct  7 14:29 proteins.txt
-rw-r--r-- 1 root root 980218 Oct  7 14:29 target-target_similarities_WS.txt
-rw-r--r-- 1 root root 240605 Oct  7 14:29 Y

===== First 3 lines of Y =====
�cnumpy.core.multiarray
_reconstruct
q cnumpy

===== Ligands file (first 300 chars) =====
{"11314340": "CC1=C2C=C(C=CC2=NN1)C3=CC(=CN=C3)OCC(CC4=CC=CC=C4)N", "24889392": "CC(C)(C)C1=CC(=NO1)NC(=O)NC2=CC=C(C=C2)C3=CN4C5=C(C=C(C=C5)OCCN6CCOCC6)SC4=N3", "11409972": "CCN1CCN(CC1)CC2=C(C=C(C=C2)NC(=O)NC3=CC=C(C=C3)OC4=NC

In [5]:
import pickle, json, numpy as np, pandas as pd

with open('/content/DeepDTA/data/davis/ligands_can.txt') as f:
    ligands = json.load(f)
smiles_list = list(ligands.values())
print(f"Num drugs: {len(smiles_list)}")

with open('/content/DeepDTA/data/davis/proteins.txt') as f:
    proteins = json.load(f)
prot_seqs = list(proteins.values())
print(f"Num proteins: {len(prot_seqs)}")

with open('/content/DeepDTA/data/davis/Y', 'rb') as f:
    Y = pickle.load(f, encoding='latin1')

print(f"Raw Y range (nM): {np.nanmin(Y):.4f} to {np.nanmax(Y):.4f}")

rows = []
for d_idx, smi in enumerate(smiles_list):
    for t_idx, seq in enumerate(prot_seqs):
        kd_nM = Y[d_idx, t_idx]
        if np.isnan(kd_nM) or kd_nM <= 0:
            continue
        # Convert nM → M → pKd
        kd_M = kd_nM * 1e-9
        pkd = -np.log10(kd_M)
        rows.append({
            'drug_id': d_idx,
            'target_id': t_idx,
            'smiles': smi,
            'sequence': seq,
            'affinity': float(pkd)
        })

df = pd.DataFrame(rows)
print(f"\nTotal interactions: {len(df)}")
print(f"pKd range: {df.affinity.min():.3f} to {df.affinity.max():.3f}")
print(df.head())

Num drugs: 68
Num proteins: 442
Raw Y range (nM): 0.0160 to 10000.0000

Total interactions: 30056
pKd range: 5.000 to 10.796
   drug_id  target_id                                             smiles  \
0        0          0  CC1=C2C=C(C=CC2=NN1)C3=CC(=CN=C3)OCC(CC4=CC=CC...   
1        0          1  CC1=C2C=C(C=CC2=NN1)C3=CC(=CN=C3)OCC(CC4=CC=CC...   
2        0          2  CC1=C2C=C(C=CC2=NN1)C3=CC(=CN=C3)OCC(CC4=CC=CC...   
3        0          3  CC1=C2C=C(C=CC2=NN1)C3=CC(=CN=C3)OCC(CC4=CC=CC...   
4        0          4  CC1=C2C=C(C=CC2=NN1)C3=CC(=CN=C3)OCC(CC4=CC=CC...   

                                            sequence  affinity  
0  MKKFFDSRREQGGSGLGSGSSGGGGSTSGLGSGYIGRVFGIGRQQV...  7.366532  
1  PFWKILNPLLERGTYYYFMGQQPGKVLGDQRRPSLPALHFIKGAGK...  5.000000  
2  PFWKILNPLLERGTYYYFMGQQPGKVLGDQRRPSLPALHFIKGAGK...  5.000000  
3  PFWKILNPLLERGTYYYFMGQQPGKVLGDQRRPSLPALHFIKGAGK...  5.000000  
4  PFWKILNPLLERGTYYYFMGQQPGKVLGDQRRPSLPALHFIKGAGK...  5.000000  


In [6]:
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from sklearn.model_selection import train_test_split

# Official DeepDTA encoding dictionaries
DTA_DRUG_DICT = {"(": 1, ".": 2, "0": 3, "2": 4, "4": 5, "6": 6, "8": 7,
    "@": 8, "B": 9, "D": 10, "F": 11, "H": 12, "L": 13, "N": 14, "P": 15,
    "R": 16, "T": 17, "V": 18, "Z": 19, "\\": 20, "b": 21, "d": 22, "f": 23,
    "h": 24, "l": 25, "n": 26, "r": 27, "t": 28, "#": 29, "%": 30, ")": 31,
    "+": 32, "-": 33, "/": 34, "1": 35, "3": 36, "5": 37, "7": 38, "9": 39,
    "=": 40, "A": 41, "C": 42, "E": 43, "G": 44, "I": 45, "K": 46, "M": 47,
    "O": 48, "S": 49, "U": 50, "W": 51, "Y": 52, "[": 53, "]": 54, "a": 55,
    "c": 56, "e": 57, "g": 58, "i": 59, "m": 60, "o": 61, "s": 62, "u": 63, "y": 64}
DTA_PROT_DICT = {"A": 1, "C": 2, "B": 3, "E": 4, "D": 5, "G": 6, "F": 7,
    "I": 8, "H": 9, "K": 10, "M": 11, "L": 12, "O": 13, "N": 14, "Q": 15,
    "P": 16, "S": 17, "R": 18, "U": 19, "T": 20, "W": 21, "V": 22, "Y": 23,
    "X": 24, "Z": 25}

MAX_SMI_LEN = 85
MAX_SEQ_LEN = 1200
NUM_SMI = 64
NUM_AA = 25

def encode_smiles(s):
    enc = [DTA_DRUG_DICT[c] for c in s if c in DTA_DRUG_DICT][:MAX_SMI_LEN]
    return enc + [0] * (MAX_SMI_LEN - len(enc))

def encode_seq(s):
    enc = [DTA_PROT_DICT[c] for c in s if c in DTA_PROT_DICT][:MAX_SEQ_LEN]
    return enc + [0] * (MAX_SEQ_LEN - len(enc))

print("Encoding...")
X_drug = np.array([encode_smiles(s) for s in df['smiles']], dtype=np.int32)
X_prot = np.array([encode_seq(s) for s in df['sequence']], dtype=np.int32)
y = df['affinity'].values.astype(np.float32)
print(f"X_drug: {X_drug.shape}, X_prot: {X_prot.shape}, y: {y.shape}")

X_d_tr, X_d_te, X_p_tr, X_p_te, y_tr, y_te = train_test_split(
    X_drug, X_prot, y, test_size=0.2, random_state=42
)
print(f"Train: {len(y_tr)}, Test: {len(y_te)}\n")

def build_deepdta():
    drug_in = layers.Input(shape=(MAX_SMI_LEN,), name="drug")
    x = layers.Embedding(NUM_SMI + 1, 128)(drug_in)
    x = layers.Conv1D(32, 4, activation='relu')(x)
    x = layers.Conv1D(64, 6, activation='relu')(x)
    x = layers.Conv1D(96, 8, activation='relu')(x)
    x = layers.GlobalMaxPooling1D()(x)

    prot_in = layers.Input(shape=(MAX_SEQ_LEN,), name="prot")
    p = layers.Embedding(NUM_AA + 1, 128)(prot_in)
    p = layers.Conv1D(32, 4, activation='relu')(p)
    p = layers.Conv1D(64, 8, activation='relu')(p)
    p = layers.Conv1D(96, 12, activation='relu')(p)
    p = layers.GlobalMaxPooling1D()(p)

    z = layers.Concatenate()([x, p])
    z = layers.Dense(1024, activation='relu')(z)
    z = layers.Dropout(0.1)(z)
    z = layers.Dense(1024, activation='relu')(z)
    z = layers.Dropout(0.1)(z)
    z = layers.Dense(512, activation='relu')(z)
    out = layers.Dense(1)(z)

    m = models.Model(inputs=[drug_in, prot_in], outputs=out)
    m.compile(optimizer=optimizers.Adam(1e-3), loss='mse')
    return m

def ci(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    n = len(y_true)
    conc = 0.0
    comp = 0.0
    for i in range(0, n, 500):
        t = y_true[i:i+500, None] - y_true[None, :]
        p = y_pred[i:i+500, None] - y_pred[None, :]
        mask = t > 0
        comp += mask.sum()
        conc += ((p > 0) & mask).sum() + 0.5 * ((p == 0) & mask).sum()
    return conc / comp if comp else 0.5

class CICB(tf.keras.callbacks.Callback):
    def __init__(self, Xd, Xp, yv):
        self.Xd, self.Xp, self.yv = Xd, Xp, yv
    def on_epoch_end(self, epoch, logs=None):
        preds = self.model.predict([self.Xd, self.Xp], batch_size=512, verbose=0).flatten()
        c = ci(self.yv, preds)
        print(f"  → val_CI: {c:.4f}  val_MSE: {np.mean((self.yv - preds)**2):.4f}")

model = build_deepdta()
model.summary()

history = model.fit(
    [X_d_tr, X_p_tr], y_tr,
    validation_data=([X_d_te, X_p_te], y_te),
    epochs=100,
    batch_size=256,
    callbacks=[CICB(X_d_te, X_p_te, y_te)],
    verbose=1
)

preds = model.predict([X_d_te, X_p_te], batch_size=512, verbose=0).flatten()
print(f"\n===== FINAL TEST RESULTS =====")
print(f"Test CI:  {ci(y_te, preds):.4f}")
print(f"Test MSE: {np.mean((y_te - preds)**2):.4f}")

Encoding...
X_drug: (30056, 85), X_prot: (30056, 1200), y: (30056,)
Train: 24044, Test: 6012



Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ drug (InputLayer)   │ (None, 85)        │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ prot (InputLayer)   │ (None, 1200)      │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 85, 128)   │      8,320 │ drug[0][0]        │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, 1200, 128) │      3,328 │ prot[0][0]        │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 82, 32)    │     16,416 │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_3 (Conv1D)   │ (None, 1197, 32)  │     16,416 │ embedding_1[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_1 (Conv1D)   │ (None, 77, 64)    │     12,352 │ conv1d[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_4 (Conv1D)   │ (None, 1190, 64)  │     16,448 │ conv1d_3[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_2 (Conv1D)   │ (None, 70, 96)    │     49,248 │ conv1d_1[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_5 (Conv1D)   │ (None, 1179, 96)  │     73,824 │ conv1d_4[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pooling… │ (None, 96)        │          0 │ conv1d_2[0][0]    │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pooling… │ (None, 96)        │          0 │ conv1d_5[0][0]    │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 192)       │          0 │ global_max_pooli… │
│ (Concatenate)       │                   │            │ global_max_pooli… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 1024)      │    197,632 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 1024)      │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 1024)      │  1,049,600 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 1024)      │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 512)       │    524,800 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 1)         │        513 │ dense_2[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 1,968,897 (7.51 MB)

 Trainable params: 1,968,897 (7.51 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 245ms/step - loss: 6.7445  → val_CI: 0.7643  val_MSE: 0.5695
94/94 ━━━━━━━━━━━━━━━━━━━━ 46s 352ms/step - loss: 2.5174 - val_loss: 0.5695
Epoch 2/100
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 149ms/step - loss: 0.6614  → val_CI: 0.7849  val_MSE: 0.5549
94/94 ━━━━━━━━━━━━━━━━━━━━ 15s 163ms/step - loss: 0.6284 - val_loss: 0.5549
Epoch 3/100
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 152ms/step - loss: 0.6094  → val_CI: 0.7944  val_MSE: 0.5368
94/94 ━━━━━━━━━━━━━━━━━━━━ 16s 166ms/step - loss: 0.5952 - val_loss: 0.5368
Epoch 4/100
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 156ms/step - loss: 0.5815  → val_CI: 0.7976  val_MSE: 0.5061
94/94 ━━━━━━━━━━━━━━━━━━━━ 16s 171ms/step - loss: 0.5806 - val_loss: 0.5061
Epoch 5/100
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 154ms/step - loss: 0.5892  → val_CI: 0.7964  val_MSE: 0.5015
94/94 ━━━━━━━━━━━━━━━━━━━━ 16s 168ms/step - loss: 0.5881 - val_loss: 0.5015
Epoch 6/100
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 152ms/step - loss: 0.5522  → val_CI: 0.8011  val_MSE: 0.4950
94

In [7]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [8]:
import os, json, numpy as np

SAVE_DIR = '/content/drive/MyDrive/deepdta_project/davis_baseline'
os.makedirs(SAVE_DIR, exist_ok=True)

# Save model
model.save(f'{SAVE_DIR}/final_model.keras')

# Save predictions (Hari will need these for metrics testing)
preds = model.predict([X_d_te, X_p_te], batch_size=512, verbose=0).flatten()
np.save(f'{SAVE_DIR}/y_test.npy', y_te)
np.save(f'{SAVE_DIR}/y_pred.npy', preds)

# Save results
results = {
    "model": "DeepDTA (CNN + CNN)",
    "dataset": "davis",
    "split": "random 80/20",
    "test_ci": 0.8911,
    "test_mse": 0.2147,
    "epochs": 100,
    "batch_size": 256,
    "learning_rate": 0.001,
    "paper_ci": 0.878,
    "paper_mse": 0.261
}
with open(f'{SAVE_DIR}/results.json', 'w') as f:
    json.dump(results, f, indent=2)

print("Saved to:", SAVE_DIR)
!ls -la {SAVE_DIR}

Saved to: /content/drive/MyDrive/deepdta_project/davis_baseline
total 23216
drwx------ 2 root root     4096 Oct  7 14:57 .
drwx------ 4 root root     4096 Oct  7 09:37 ..
-rw------- 1 root root 23715239 Oct  7 14:57 final_model.keras
-rw------- 1 root root      235 Oct  7 14:57 results.json
-rw------- 1 root root    24176 Oct  7 14:57 y_pred.npy
-rw------- 1 root root    24176 Oct  7 14:57 y_test.npy


In [9]:
import pickle, json, numpy as np, pandas as pd, os, tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from sklearn.model_selection import train_test_split

# ============ LOAD KIBA DATA ============
with open('/content/DeepDTA/data/kiba/ligands_can.txt') as f:
    ligands = json.load(f)
smiles_list = list(ligands.values())

with open('/content/DeepDTA/data/kiba/proteins.txt') as f:
    proteins = json.load(f)
prot_seqs = list(proteins.values())

with open('/content/DeepDTA/data/kiba/Y', 'rb') as f:
    Y = pickle.load(f, encoding='latin1')

print(f"Y shape: {Y.shape}, range: {np.nanmin(Y):.3f} to {np.nanmax(Y):.3f}")

# KIBA: use score directly (no log transform)
rows = []
for d_idx, smi in enumerate(smiles_list):
    for t_idx, seq in enumerate(prot_seqs):
        val = Y[d_idx, t_idx]
        if np.isnan(val):
            continue
        rows.append({'smiles': smi, 'sequence': seq, 'affinity': float(val)})

df = pd.DataFrame(rows)
print(f"Loaded {len(df)} interactions")
print(f"Affinity range: {df.affinity.min():.3f} to {df.affinity.max():.3f}")

# ============ ENCODING ============
DTA_DRUG_DICT = {"(": 1, ".": 2, "0": 3, "2": 4, "4": 5, "6": 6, "8": 7,
    "@": 8, "B": 9, "D": 10, "F": 11, "H": 12, "L": 13, "N": 14, "P": 15,
    "R": 16, "T": 17, "V": 18, "Z": 19, "\\": 20, "b": 21, "d": 22, "f": 23,
    "h": 24, "l": 25, "n": 26, "r": 27, "t": 28, "#": 29, "%": 30, ")": 31,
    "+": 32, "-": 33, "/": 34, "1": 35, "3": 36, "5": 37, "7": 38, "9": 39,
    "=": 40, "A": 41, "C": 42, "E": 43, "G": 44, "I": 45, "K": 46, "M": 47,
    "O": 48, "S": 49, "U": 50, "W": 51, "Y": 52, "[": 53, "]": 54, "a": 55,
    "c": 56, "e": 57, "g": 58, "i": 59, "m": 60, "o": 61, "s": 62, "u": 63, "y": 64}
DTA_PROT_DICT = {"A": 1, "C": 2, "B": 3, "E": 4, "D": 5, "G": 6, "F": 7,
    "I": 8, "H": 9, "K": 10, "M": 11, "L": 12, "O": 13, "N": 14, "Q": 15,
    "P": 16, "S": 17, "R": 18, "U": 19, "T": 20, "W": 21, "V": 22, "Y": 23,
    "X": 24, "Z": 25}

MAX_SMI_LEN, MAX_SEQ_LEN = 100, 1000
NUM_SMI, NUM_AA = 64, 25

def enc_smi(s):
    e = [DTA_DRUG_DICT[c] for c in s if c in DTA_DRUG_DICT][:MAX_SMI_LEN]
    return e + [0] * (MAX_SMI_LEN - len(e))

def enc_seq(s):
    e = [DTA_PROT_DICT[c] for c in s if c in DTA_PROT_DICT][:MAX_SEQ_LEN]
    return e + [0] * (MAX_SEQ_LEN - len(e))

print("Encoding drug SMILES...")
X_drug = np.array([enc_smi(s) for s in df['smiles']], dtype=np.int32)
print("Encoding protein sequences...")
X_prot = np.array([enc_seq(s) for s in df['sequence']], dtype=np.int32)
y = df['affinity'].values.astype(np.float32)
print(f"X_drug: {X_drug.shape}, X_prot: {X_prot.shape}, y: {y.shape}")

X_d_tr, X_d_te, X_p_tr, X_p_te, y_tr, y_te = train_test_split(
    X_drug, X_prot, y, test_size=0.2, random_state=42
)
print(f"Train: {len(y_tr)}, Test: {len(y_te)}")

# ============ MODEL ============
def build_deepdta():
    drug_in = layers.Input(shape=(MAX_SMI_LEN,), name="drug")
    x = layers.Embedding(NUM_SMI + 1, 128)(drug_in)
    x = layers.Conv1D(32, 4, activation='relu')(x)
    x = layers.Conv1D(64, 6, activation='relu')(x)
    x = layers.Conv1D(96, 8, activation='relu')(x)
    x = layers.GlobalMaxPooling1D()(x)

    prot_in = layers.Input(shape=(MAX_SEQ_LEN,), name="prot")
    p = layers.Embedding(NUM_AA + 1, 128)(prot_in)
    p = layers.Conv1D(32, 4, activation='relu')(p)
    p = layers.Conv1D(64, 8, activation='relu')(p)
    p = layers.Conv1D(96, 12, activation='relu')(p)
    p = layers.GlobalMaxPooling1D()(p)

    z = layers.Concatenate()([x, p])
    z = layers.Dense(1024, activation='relu')(z)
    z = layers.Dropout(0.1)(z)
    z = layers.Dense(1024, activation='relu')(z)
    z = layers.Dropout(0.1)(z)
    z = layers.Dense(512, activation='relu')(z)
    out = layers.Dense(1)(z)
    m = models.Model(inputs=[drug_in, prot_in], outputs=out)
    m.compile(optimizer=optimizers.Adam(1e-3), loss='mse')
    return m

def ci(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    n = len(y_true); conc = 0.0; comp = 0.0
    for i in range(0, n, 500):
        t = y_true[i:i+500, None] - y_true[None, :]
        p = y_pred[i:i+500, None] - y_pred[None, :]
        mask = t > 0
        comp += mask.sum()
        conc += ((p > 0) & mask).sum() + 0.5 * ((p == 0) & mask).sum()
    return conc / comp if comp else 0.5

class CICB(tf.keras.callbacks.Callback):
    def __init__(self, Xd, Xp, yv, save_dir):
        self.Xd, self.Xp, self.yv, self.save_dir = Xd, Xp, yv, save_dir
        self.best_ci = 0.0
    def on_epoch_end(self, epoch, logs=None):
        preds = self.model.predict([self.Xd, self.Xp], batch_size=512, verbose=0).flatten()
        c = ci(self.yv, preds)
        if c > self.best_ci:
            self.best_ci = c
            self.model.save(f'{self.save_dir}/best_model.keras')
        print(f"  → val_CI: {c:.4f}  val_MSE: {np.mean((self.yv - preds)**2):.4f}")

# ============ TRAIN ============
SAVE_DIR = '/content/drive/MyDrive/deepdta_project/kiba_baseline'
os.makedirs(SAVE_DIR, exist_ok=True)

model = build_deepdta()
history = model.fit(
    [X_d_tr, X_p_tr], y_tr,
    validation_data=([X_d_te, X_p_te], y_te),
    epochs=100,
    batch_size=256,
    callbacks=[CICB(X_d_te, X_p_te, y_te, SAVE_DIR)],
    verbose=1
)

# ============ FINAL EVAL + SAVE ============
preds = model.predict([X_d_te, X_p_te], batch_size=512, verbose=0).flatten()
test_ci = ci(y_te, preds)
test_mse = float(np.mean((y_te - preds)**2))

model.save(f'{SAVE_DIR}/final_model.keras')
np.save(f'{SAVE_DIR}/y_test.npy', y_te)
np.save(f'{SAVE_DIR}/y_pred.npy', preds)

results = {
    "model": "DeepDTA (CNN + CNN)",
    "dataset": "kiba",
    "split": "random 80/20",
    "test_ci": round(test_ci, 4),
    "test_mse": round(test_mse, 4),
    "epochs": 100, "batch_size": 256, "learning_rate": 0.001,
    "paper_ci": 0.863, "paper_mse": 0.194
}
with open(f'{SAVE_DIR}/results.json', 'w') as f:
    json.dump(results, f, indent=2)

print(f"\n===== KIBA DONE =====")
print(f"Test CI:  {test_ci:.4f}  (paper: 0.863)")
print(f"Test MSE: {test_mse:.4f}  (paper: 0.194)")
print(f"\nSaved to: {SAVE_DIR}")
!ls -la {SAVE_DIR}

Y shape: (2111, 229), range: 0.000 to 17.200
Loaded 118254 interactions
Affinity range: 0.000 to 17.200
Encoding drug SMILES...
Encoding protein sequences...
X_drug: (118254, 100), X_prot: (118254, 1000), y: (118254,)
Train: 94603, Test: 23651
Epoch 1/100
370/370 ━━━━━━━━━━━━━━━━━━━━ 0s 153ms/step - loss: 9.4649  → val_CI: 0.7231  val_MSE: 0.4995
370/370 ━━━━━━━━━━━━━━━━━━━━ 86s 200ms/step - loss: 2.4843 - val_loss: 0.4995
Epoch 2/100
370/370 ━━━━━━━━━━━━━━━━━━━━ 0s 131ms/step - loss: 0.6021  → val_CI: 0.7482  val_MSE: 0.5662
370/370 ━━━━━━━━━━━━━━━━━━━━ 57s 154ms/step - loss: 0.5877 - val_loss: 0.5662
Epoch 3/100
370/370 ━━━━━━━━━━━━━━━━━━━━ 0s 129ms/step - loss: 0.5477  → val_CI: 0.7548  val_MSE: 0.4225
370/370 ━━━━━━━━━━━━━━━━━━━━ 57s 153ms/step - loss: 0.5488 - val_loss: 0.4225
Epoch 4/100
370/370 ━━━━━━━━━━━━━━━━━━━━ 0s 130ms/step - loss: 0.5342  → val_CI: 0.7612  val_MSE: 0.4395
370/370 ━━━━━━━━━━━━━━━━━━━━ 57s 154ms/step - loss: 0.5414 - val_loss: 0.4395
Epoch 5/100
370/370 ━━━━